In [0]:
orders_stage= "eccom_catalog.default.orders_stage"
customers_stage= "eccom_catalog.default.customers_stage"
product_stage= "eccom_catalog.default.product_stage"
inventory_stage= "eccom_catalog.default.inventory_stage"
shipping_stage= "eccom_catalog.default.shipping_stage"
enriched_orders_table= "eccom_catalog.default.enriched_orders"
customer_analytics_table= "eccom_catalog.default.customer_analytics"
product_analytics= "eccom_catalog.default.product_analytics"
print("Starting data enrichment process...")

In [0]:
from pyspark.sql.functions import current_timestamp,when,dayofweek,count,sum,min,avg,countDistinct,datediff,current_date,col,lit,hour
from datetime import datetime
#Read all staging tables 
try:
	df_orders=spark.read.table(orders_stage)
	df_customers=spark.read.table(customers_stage)
	df_product=spark.read.table(product_stage)
	df_inventory=spark.read.table(inventory_stage)
	df_shipping=spark.read.table(shipping_stage)
	print("Successfully loaded all staging tables for enrichment")

except Exception as e:
	print(f"Error reading staging tables: {str(e)}")
	raise

In [0]:
#Created enriched orders dataset with all related information
from pyspark.sql.functions import day
try:
	#Rename all conflicting columns to avoid ambiguity
	df_customers=df_customers.withColumnRenamed("created_timestamp", "customer_created_timestamp").withColumnRenamed("batch_id", "customer_batch_id")\
	.withColumnRenamed("processed_timestamp", "customer_processed_timestamp").withColumnRenamed("source_system", "customer_source_system")\
	.withColumnRenamed("lifecycle_stage","customer_lifecycle_stage") 
	 
	df_product=df_product.withColumnRenamed("created_timestamp", "product_created_timestamp").withColumnRenamed("batch_id", "product_batch_id")\
	.withColumnRenamed("processed_timestamp", "product_processed_timestamp").withColumnRenamed("source_system", "product_source_system")\
	.withColumnRenamed("lifecycle_stage", "product_lifecycle_stage").withColumnRenamed("currency", "product_currency")\
    .withColumnRenamed("weight_kg", "product_weight_kg")\
	.withColumnRenamed("stock_quantity", "product_stock_quantity").withColumnRenamed("stock_status", "product_stock_status")\
	.withColumnRenamed("price_segment", "product_price_segment")

	df_inventory=df_inventory.withColumnRenamed("created_timestamp", "inventory_created_timestamp").withColumnRenamed("batch_id", "inventory_batch_id")\
	.withColumnRenamed("processed_timestamp", "inventory_processed_timestamp").withColumnRenamed("source_system", "inventory_source_system")\
	.withColumnRenamed("stock_status", "inventory_stock_status")

	df_shipping=df_shipping.withColumnRenamed("created_timestamp", "shipping_created_timestamp").withColumnRenamed("batch_id", "shipping_batch_id")\
	.withColumnRenamed("processed_timestamp", "shipping_processed_timestamp").withColumnRenamed("source_system", "shipping_source_system")\
	.withColumnRenamed("currency", "shipping_currency").withColumnRenamed("package_weight", "shipping_package_weight")

	#Join orders with customers, products, inventory and shipping 
	df_enriched_orders=df_orders.join(df_customers,on="customer_id",how="left").join(df_product,on="product_id",how="left")\
	.join(df_inventory, on="product_id",how="left").join(df_shipping,on="order_id",how="left")

	#Add business metrices
	#Add 30% profit margin
	df_enriched_orders=df_enriched_orders.withColumn("order_profit_margin", col("order_amount")*0.3)

	#Add customer lifecycle value estimation
	df_enriched_orders=df_enriched_orders.withColumn("estimated_clv", when(col("customer_tier")=="premium", col("order_amount")*10) 
																	   .when(col("customer_tier")=="gold", col("order_amount")*7) 
																	   .when(col("customer_tier")=="silver", col("order_amount")*5) 
																	   .otherwise(col("order_amount")*3))
																	
	#Add seasonal indicators
	df_enriched_orders=df_enriched_orders.withColumn("season", when(day(col("order_date")).isin([12,1,2]), "Winter") 
															   .when(day(col("order_date")).isin([3,4,5]), "Spring") 
															   .when(day(col("order_date")).isin([6,7,8]), "Summer") 
															   .otherwise("Fail"))

	#Add day of week and time of day indicators
	df_enriched_orders=df_enriched_orders.withColumn("day_of_week", dayofweek(col("order_date")))\
	.withColumn("is_weekend", when(col("day_of_week").isin([1,7]), True).otherwise(False))\
	.withColumn("time_of_day", when(hour(col("created_timestamp"))<6, "Early Morning") 
							   .when(hour(col("created_timestamp"))<12, "Morning") 
							   .when(hour(col("created_timestamp"))<18, "Afternoon")
							   .otherwise("Evening"))
	print("Enriched orders dataset created successfully")

except Exception as e:
	print(f"Failed to create enriched orders dataset: {str(e)}")
	raise

In [0]:
#Create customer analytics dataset
from pyspark.sql.functions import max
try:
	df_customer_analytics=df_enriched_orders.groupBy("customer_id").agg(count("order_id").alias("total_orders"),sum("order_amount").alias("total_spent"),
	avg("order_amount").alias("average_order_amount"),min("order_date").alias("first_order_date"),max("order_date").alias("last_order_date"),countDistinct("product_id").alias("unique_products_purchased"),
	countDistinct("category").alias("unique_categories_purchased"),sum("order_profit_margin").alias("total_profit_generated"),
	avg("estimated_clv").alias("avg_estimated_clv"))

	#Join with customer details
	df_customer_analytics=df_customer_analytics.join(df_customers, on="customer_id",how="left")

	#Calculate additional metrices
	df_customer_analytics=df_customer_analytics.withColumn("days_since_first_order", datediff(current_date(), col("first_order_date")))\
	.withColumn("days_since_last_order", datediff(current_date(), col("last_order_date")))\
	.withColumn("order_frequency_days", col("days_since_first_order")/col("total_orders"))

	#Create customer segments
	df_customer_analytics=df_customer_analytics.withColumn("customer_segment", when((col("total_spent")>=100) & (col("total_orders")>=5), "VIP").when((col("total_spent")>=500) & (col("total_orders")>=3), "High Value") 
	.when((col("total_spent")>=200) & (col("total_orders")>=2), "Medium Value") 
    .otherwise("Low Value"))
																			   
	#Create customer lifecycle stage
	df_customer_analytics=df_customer_analytics.withColumn("lifecycle_stage", when(col("days_since_last_order")<=30, "Active") 
																			  .when(col("days_since_last_order")<=90, "At Risk") 
																			  .when(col("days_since_last_order")<100, "Inactive") 
																			  .otherwise("Lost"))
	print("Customer analytics dataset created successfully")

except Exception as e:
	print(f"Failed to create customer analytics dataset: {str(e)}")
	raise

In [0]:
#Create product analytics dataset 
try:
	#Calculate product metrices 
	df_product_analytics=df_enriched_orders.groupBy("product_id").agg(count("order_id").alias("total_orders"), sum("order_amount").alias("total_revenue"),
	avg("order_amount").alias("average_order_amount"), countDistinct("customer_id").alias("unique_customers"), sum("order_profit_margin").alias("total_profit"),
	min("order_date").alias("first_order_date"), max("order_date").alias("last_order_date"))
	
	#Join with product details
	df_product_analytics=df_product_analytics.join(df_product,on="product_id",how="left")

	#Calculate additional metrices
	df_product_analytics=df_product_analytics.withColumn("days_since_first_order", datediff(current_date(), col("first_order_date")))\
     .withColumn("days_since_last_order", datediff(current_date(), col("last_order_date")))\
	 .withColumn("order_frequency_days", col("days_since_first_order")/col("total_orders"))\
	 .withColumn("revenue_per_customer", col("total_revenue")/col("unique_customers"))

	#Create product performance categories
	df_product_analytics=df_product_analytics.withColumn("performance_category", when((col("total_revenue")>=5000) & (col("total_orders")>=20), "Star") 
	.when((col("total_revenue")>=2000) & (col("total_orders")>=10), "High Performer") 
	.when((col("total_revenue")>=50) & (col("total_orders")>=5), "Medium Performer") 
	.otherwise("Low Performer"))

	#Create product lifecycle stage
	df_product_analytics=df_product_analytics.withColumn("product_lifecycle", when(col("days_since_last_order")<=30, "Active") 
																			  .when(col("days_since_last_order")<=90, "Declining")
																			  .when(col("discontinued").cast("boolean")==True, "Discontinued")
																			  .otherwise("Stagnant"))
	print("Product analytics dataset created successfully")

except Exception as e:
	print(f"Error to create product analytics: {str(e)}")
	raise

In [0]:
#Write enriched datasets to tables
try:
	#Write enriched orders
	df_enriched_orders.write.format("delta").mode("overwrite").saveAsTable(enriched_orders_table)
	print(f"Successfully write enriched orders to {enriched_orders_table}")

	#Write customer analytics
	df_customer_analytics.write.format("delta").mode("overwrite").saveAsTable(customer_analytics_table)
	print(f"Successfully write customer analytics to {customer_analytics_table}")

	#Write product analytics
	df_product_analytics.write.format("delta").mode("overwrite").saveAsTable(product_analytics)
	print(f"Successfully write customer analytics to {product_analytics}")

	from pyspark.sql.types import StructType, StructField, LongType, StringType
	#Log enrichment statistics
	import json
	enrichment_summary={
	"archived_files": None,
	"invalid_records": None,
	"status": "SUCCESS",
	"task": "data_enrichment",
	"timestamp": datetime.now().isoformat(),
	"total_records": None,
	"valid_records": None 
	}	
	print(f"Enrichment Summary: {json.dumps(enrichment_summary)}")

	log_schema=StructType([
	StructField("archived_files", LongType(), True),
	StructField("invalid_records", LongType(), True),
	StructField("status", StringType(), True),
	StructField("task", StringType(), True),
	StructField("timestamp", StringType(), True),
	StructField("total_records", LongType(), True),
	StructField("valid_records", LongType(), True)
	])
	df_enrichment_summary=spark.createDataFrame([enrichment_summary], schema=log_schema)
	df_enrichment_summary.write.format("delta").mode("append").saveAsTable("eccom_catalog.default.processing_log")

	dbutils.jobs.taskValues.set("enrichment_status", "SUCCESS")

except Exception as e:
	print(f"Error writing enriched datasets: {str(e)}")
	raise

In [0]:
from pyspark.sql.types import StructType, StructField, LongType, StringType
#Log enrichment statistics
import json
enrichment_summary={
"archived_files": None,
"invalid_records": None,
"status": "SUCCESS",
"task": "data_enrichment",
"timestamp": datetime.now().isoformat(),
"total_records": None,
"valid_records": None 
}	
print(f"Enrichment Summary: {json.dumps(enrichment_summary)}")

log_schema=StructType([
StructField("archived_files", LongType(), True),
StructField("invalid_records", LongType(), True),
StructField("status", StringType(), True),
StructField("task", StringType(), True),
StructField("timestamp", StringType(), True),
StructField("total_records", LongType(), True),
StructField("valid_records", LongType(), True)
])
df_enrichment_summary=spark.createDataFrame([enrichment_summary], schema=log_schema)
df_enrichment_summary.write.format("delta").mode("append").saveAsTable("eccom_catalog.default.processing_log")

#Set validation result for downstream tasks
try:
    dbutils.jobs.taskValues.set("enrichment_status", "SUCCESS")
except Exception as e:
	print(f"Warning: Could not set task values (non-critical): {e}")    